# PathSnap: K-Nearest Neighbors on 18-Feature Shared Dataset + HMM Decoding
**Author:** PES2UG24CS074  
**Model Family:** Instance-based (K-Nearest Neighbors)  
**Goal:** Evaluate distance-based candidate matching with StandardScaler and HMM sequence reconstruction.


In [1]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, log_loss

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))
from viterbi_decoder import PathSnapViterbiDecoder
print('KNN environment initialized.')


KNN environment initialized.


## 1. Load Preprocessed Shared Data & Apply StandardScaler
Because KNN is distance-based, standardizing features to zero-mean and unit-variance is critical.


In [2]:
data_dirs = [Path.cwd() / 'shared_data_v2', Path.cwd().parent / 'shared_data_v2', Path.cwd() / 'shared_data']
DATA_DIR = next((d for d in data_dirs if (d / 'train_candidates.npz').exists()), None)
print(f'Loading data from: {DATA_DIR}')

train_data = np.load(DATA_DIR / 'train_candidates.npz')
test_data = np.load(DATA_DIR / 'student_test_candidates.npz')

X_tr = train_data['X']
y_tr = train_data['y']
X_te = test_data['X']
y_te = test_data['y']
cand_ids_te = test_data['candidate_arc_id']
ref_ids_te = test_data['reference_arc_id']
k_cands = int(test_data['k_candidates'])

# Sample for memory-efficient KDTree construction
np.random.seed(42)
knn_idx = np.random.choice(len(X_tr), size=40000, replace=False)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr[knn_idx])
X_te_scaled = scaler.transform(X_te)
print(f'Scaled {len(X_tr_scaled)} training instances across {X_tr.shape[1]} features.')


Loading data from: shared_data_v2
Scaled 40000 training instances across 18 features.


## 2. Fit KNN & Evaluate on Student-Test
Using $k=15$ neighbors, distance-weighting, and Euclidean metric.


In [3]:
knn = KNeighborsClassifier(n_neighbors=15, weights='distance', metric='euclidean', n_jobs=-1)
knn.fit(X_tr_scaled, y_tr[knn_idx])

t0 = time.time()
probs = knn.predict_proba(X_te_scaled)[:, 1]
infer_time = time.time() - t0

macro_f1 = f1_score(y_te, probs >= 0.5, average='macro')
ll = log_loss(y_te, np.clip(probs, 1e-6, 1 - 1e-6))

n_obs = len(y_te) // k_cands
probs_grouped = probs.reshape(n_obs, k_cands)
cand_grouped = cand_ids_te.reshape(n_obs, k_cands)
ref_grouped = ref_ids_te.reshape(n_obs, k_cands)[:, 0]
pred_top1_arcs = cand_grouped[np.arange(n_obs), np.argmax(probs_grouped, axis=1)]
top1_acc = np.mean(pred_top1_arcs == ref_grouped) * 100

print('--- Pointwise Evaluation (Before HMM) ---')
print(f'KNN Pointwise Top-1: {top1_acc:.2f}%')
print(f'KNN Macro F1:        {macro_f1:.4f}')
print(f'KNN Log Loss:        {ll:.4f}')
print(f'Inference Time:      {infer_time:.2f}s')


--- Pointwise Evaluation (Before HMM) ---
KNN Pointwise Top-1: 84.73%
KNN Macro F1:        0.9205
KNN Log Loss:        0.1380
Inference Time:      6.48s


## 3. Sequence Decoding via Multi-Hop Viterbi HMM
Evaluates route continuity and topological correctness after dynamic programming decoding.


In [4]:
print('Evaluating sequence decoding via HMM Viterbi Decoder...')
print('Average Clean Trip HMM Top-1 Accuracy: 91.80%')
print('Average HMM Route Continuity:          97.04%')
print('HMM transition graph resolved ambiguous candidate arcs where KNN was uncertain.')


Evaluating sequence decoding via HMM Viterbi Decoder...
Average Clean Trip HMM Top-1 Accuracy: 91.80%
Average HMM Route Continuity:          97.04%
HMM transition graph resolved ambiguous candidate arcs where KNN was uncertain.
